In [13]:
import os
from pathlib import Path
import sys

os.chdir(Path(__file__).parent.parent if '__file__' in dir() else Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
print("Working directory:", Path.cwd())

Working directory: /Users/jimanderssen/Library/CloudStorage/OneDrive-Mittuniversitetet/data_repositories/mfa_project


In [23]:
import pandas as pd
import numpy as np
import eurostat
from src.loaders import load_dataset, extend_eurostat_dataset
from src.utils import smart_format
%load_ext autoreload
%autoreload 2
pd.options.display.max_columns = 999
pd.options.display.max_rows = 150
pd.options.display.float_format = smart_format
from src.loaders.eprtr_emissions import load_all_emissions



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
wasgen = extend_eurostat_dataset(load_dataset('env_wasgen'), ["nace_r2", "waste", "geo"])
emissions = load_all_emissions('data/raw')

Loaded 353,580 air release records


/Users/jimanderssen/Library/CloudStorage/OneDrive-Mittuniversitetet/data_repositories/mfa_project/src/loaders/eprtr_emissions.py:53: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Loaded 240,794 water release records
Loaded 61,071 transfer records
Total: 655,445 emission records from 34,189 facilities


# E-PRTR emissions data example

### SSAB Luleå - Air pollutant emissions (2022)

In [25]:
emissions[(emissions['facility_name'].str.contains('SSAB EMEA AB i Luleå',na=False))&(emissions['medium']=='AIR')].pivot_table(
    columns='reporting_year',index='pollutant',values='release_kg')[2022].reset_index().sort_values(by=2022,ascending=False)

,pollutant,2022
1,Carbon dioxide (CO2),"1,560.00M"
2,Carbon dioxide (CO2) excluding biomass,"1,560.00M"
3,Carbon monoxide (CO),7.79M
14,Sulphur oxides (SOX),"464,000"
11,Nitrogen oxides (NOX),"357,000"
13,Particulate matter (PM10),"141,000"
9,Naphthalene,"3,830"
15,Zinc and compounds (as Zn),"1,500"
8,Lead and compounds (as Pb),272.00
5,Copper and compounds (as Cu),151.00


Use classifier of some sort to go from emissions fingerprint -> process tech 
- SSAB Luleå uses BF-BOF process tech! (High CO)


- Use t slag/t CO2 to quantify waste (from BAT)
- Use share of national * NACE CO2 to allocate waste to facilities


# Eurostat national × NACE waste generation statistics

Definition of waste per Eurostat: 
- "Any substance or object which the holder discards or intends or is required to discard."

- I.e All generated waste that's being reported and classified as waste (not SRM or by-products)

-> Larger shares of reported waste to total estimated process wastes = Lower recovery maturity = Interesting for Ragn-sells!


In [54]:
wasgen[(wasgen['hazard']=='HAZ_NHAZ')&(wasgen['nace_r2']=='C24_C25')&(wasgen['waste']=='W124')&(wasgen['geo'].isin(['SE','FI','PL','DE','IT']))][['unit','geo_description','nace_r2','nace_r2_description','waste','waste_description','2004','2006','2008','2010','2012','2014','2016','2018','2020','2022']]

,unit,geo_description,nace_r2,nace_r2_description,waste,waste_description,2004,2006,2008,2010,2012,2014,2016,2018,2020,2022
59448,T,Germany,C24_C25,Manufacture of basic metals and fabricated met...,W124,Combustion wastes,8.09M,8.15M,7.29M,6.35M,4.10M,4.49M,3.66M,3.32M,2.91M,3.05M
59455,T,Finland,C24_C25,Manufacture of basic metals and fabricated met...,W124,Combustion wastes,"935,882",1.03M,"933,923","732,419",1.11M,"694,385","545,726","521,365","532,054","575,798"
59461,T,Italy,C24_C25,Manufacture of basic metals and fabricated met...,W124,Combustion wastes,6.12M,4.85M,6.08M,4.58M,4.69M,4.23M,4.30M,4.20M,4.77M,4.09M
59471,T,Poland,C24_C25,Manufacture of basic metals and fabricated met...,W124,Combustion wastes,5.74M,5.46M,5.62M,5.17M,6.34M,5.89M,5.37M,5.75M,5.45M,5.08M
59475,T,Sweden,C24_C25,Manufacture of basic metals and fabricated met...,W124,Combustion wastes,2.27M,1.79M,1.11M,"695,401","841,617","923,272",1.02M,"505,498","420,908","548,393"


Example to verify that waste definition:
- Sweden has 1,18M reported total waste for C24_C25. SSAB Luleå environmental report state they generate roughly 1,1 million/year process waste (waste+by-products)

- -> Not all waste end up in this dataset, but high reported amounts with relative to estimated waste generation should indicate no existing recovery pathways. 



In [35]:
wasgen[(wasgen['hazard']=='HAZ_NHAZ')&(wasgen['nace_r2']=='C24_C25')&(wasgen['waste']=='TOTAL')&(wasgen['geo'].isin(['SE','FI','PL','DE','IT']))][['unit','nace_r2','nace_r2_description','waste','waste_description','geo_description','2004','2006','2008','2010','2012','2014','2016','2018','2020','2022']]

,unit,nace_r2,nace_r2_description,waste,waste_description,geo_description,2004,2006,2008,2010,2012,2014,2016,2018,2020,2022
57990,T,C24_C25,Manufacture of basic metals and fabricated met...,TOTAL,Total waste,Germany,13.49M,14.08M,13.46M,11.93M,9.60M,10.77M,9.58M,9.31M,9.45M,8.80M
57997,T,C24_C25,Manufacture of basic metals and fabricated met...,TOTAL,Total waste,Finland,1.71M,1.86M,1.78M,2.16M,2.28M,1.98M,1.72M,1.32M,1.88M,1.72M
58003,T,C24_C25,Manufacture of basic metals and fabricated met...,TOTAL,Total waste,Italy,12.05M,11.94M,14.90M,10.18M,10.62M,10.18M,10.79M,10.80M,10.79M,11.00M
58013,T,C24_C25,Manufacture of basic metals and fabricated met...,TOTAL,Total waste,Poland,41.63M,40.14M,37.49M,9.02M,10.79M,10.91M,10.29M,11.84M,9.51M,9.33M
58017,T,C24_C25,Manufacture of basic metals and fabricated met...,TOTAL,Total waste,Sweden,4.86M,2.93M,2.36M,1.69M,1.61M,1.54M,1.70M,1.10M,1.04M,1.18M


Trend analysis can be included to follow recovery maturity developments in countries

# Comparison: Eurostat waste generation data and E-PRTR off-site waste transfers

In [48]:
multi_summary_table = pd.read_csv('data/processed/eprtr_wasgen_comparison/multi_C24_C25_summary.csv')
multi_table = pd.read_csv('data/processed/eprtr_wasgen_comparison/multi_C24_C25.csv')

### Swedish nace C24_C25

Big ratio difference could indicate few recovery pathways for waste.

In [53]:
multi_table[multi_table['country']=='SE'][['year','eprtr_tonnes','wasgen_tonnes','ratio']]

,year,eprtr_tonnes,wasgen_tonnes,ratio
0,2008,"773,675",2.36M,0.33
1,2010,"859,368",1.69M,0.51
2,2012,"812,396",1.61M,0.50
3,2014,"846,066",1.54M,0.55
4,2016,"852,195",1.70M,0.50
5,2018,"926,761",1.10M,0.84
6,2020,"827,328",1.04M,0.79
7,2022,"912,104",1.18M,0.78


In [57]:
multi_table[multi_table['country']=='PL'][['year','eprtr_tonnes','wasgen_tonnes','ratio']]

,year,eprtr_tonnes,wasgen_tonnes,ratio
8,2008,4.70M,37.49M,0.13
9,2010,4.64M,9.02M,0.51
10,2012,4.47M,10.79M,0.41
11,2014,4.83M,10.91M,0.44
12,2016,5.63M,10.29M,0.55
13,2018,5.28M,11.84M,0.45
14,2020,5.57M,9.51M,0.59
15,2022,4.75M,9.33M,0.51


### Coverage ratio of off-site waste transfers and reported waste generation

In [46]:
multi_summary_table

,Unnamed: 0,Germany,Finland,Poland,Sweden
0,Mean coverage ratio,96%,56%,49%,60%
1,Ratio trend,Rising (82%->106%),Declining (97%->53%),Stable (51%->51%),Rising (33%->78%)
2,env_wasgen trend,Declining (13.5M->8.8M),Stable (1.8M->1.7M),Stable (9.0M->9.3M),Declining (2.4M->1.2M)
3,E-PRTR trend,Stable (11.0M->9.3M),Declining (1.7M->915k),Stable (4.6M->4.8M),Stable (774k->912k)


All process waste is likely not covered in between these report systems, but the ratio differences can help to indicate lacking coverage.
- Sweden's coverage ratio would be lower if facilities like SSAB Luleå didn't recover their process waste as well as they do! should be higher if they